In [ ]:
#from 'epr of networks.ipynb'
#build null-model (shuffle IsIs); needs spike-times
np.random.seed(42)

def shuffle_isi(spikes):
    spikesH0 = np.zeros([*spikes.shape])
    sp_timesH0, sp_neuronH0 = spikes[0], spikes[1] #shuf_test[0], shuf_test[1]
    num_neurons = np.unique(sp_neuronH0).size

    level = 0
    for i_neuron in range(num_neurons):#shuffle individually for each neuron; or should I just shuffle the neuron's indices?
        times = sp_timesH0[sp_neuronH0==i_neuron]
        #include time 0 for propper shuffling of isis
        if times[0]==0: #...except for the case, where 1st time is 0 already; TODO does size match then?
            isis = np.diff(times)
            print('spike at time 0 for neuron {}; shuffling might not be entirely random in that case'.format(i_neuron))
        else:
            times0 = np.zeros([times.size+1]) 
            times0[1:] = times
            #get isis
            isis = np.diff(times0)
        #shuffle isis
        np.random.shuffle(isis)
        shuffled_times = np.cumsum(isis) #get new timepoints from shuffled isis
        #print(times.size==shuffled_times.size)
        #add to new spike array
        nspikes_neuron_i = times.size
        spikesH0[0, level : level+nspikes_neuron_i] = shuffled_times
        spikesH0[1, level : level+nspikes_neuron_i] = np.repeat(i_neuron, nspikes_neuron_i)
        level+= nspikes_neuron_i 
    #sort array by ascending spike-times
    spikesH0 = spikesH0[:,spikesH0[0,:].argsort()]
    return spikesH0
#spikesH0 = shuffle_isi(pop_sp_times)
# #bin
# print('binning...')
# time_bins, binned_spikesH0 = bin_tp_data(spikesH0, T=T, dt=dt)
# #filter
# print('filtering...')
# pop_sprateH0 = inst_rate(binned_spikesH0, filter_width= filter_width, dt=dt)

def Poisson_model(binned_spikes): #TODO: check fano factor, for > 1 spike/per_bin
    '''takes the rates of each neuron over the whole trajectory and generates new
    trajectory from poisson-process. EPR should be close to 0 -> negative ctrl'''
    nNeurons, nbins = binned_spikes.shape
    rates = binned_spikes.sum(axis=1) #spikes/T -> np's poisson takes no time-interval, so it needs to be in rate
    nspikes = np.random.poisson(rates)

    pois_model = np.zeros([*binned_spikes.shape])
    for ni in range(nNeurons):
        rn_ind = np.random.randint(0, nbins, nspikes[ni])
        pois_model[ni, rn_ind] = 1 
    #check, whether data binomial
    if np.any(binned_spikes>1)|np.any(pois_model>1):
        print('Warning: data wasnt bernoulli/binary due to binning, so Fano-factor might be incorrect')

    mean_data, mean_pois = np.mean(binned_spikes, axis=1), np.mean(pois_model, axis=1)
    var_data, var_pois = mean_data*(1 - mean_data), mean_pois*(1 - mean_pois)

    FF_data = var_data/mean_data
    FF_model = var_pois/mean_pois

    print('mean fano-factors:', np.mean(FF_data), 'for data, and ', np.mean(FF_model), 'for model')
    # plt.figure(figsize=(15,3))
    # plt.bar(np.arange(nNeurons), FF_data,label='data')
    # plt.bar(np.arange(nNeurons), FF_model,label='model',alpha = 0.5)
    # plt.ylabel('FF')
    # plt.xlabel('neuron id')
    # plt.legend()
    return pois_model


In [ ]:
#generate feed-forward network activity
np.random.seed(42)

def feedforward_generator(ndim=3, T=10000, len_period=9, delay=3, dt=1, noise_std=0):
    ntp = int(T/dt)
    ntp_period = int(len_period/dt)
    ntp_delay = int(delay/dt)

    period = np.zeros([ntp_period])
    period[0] = 1

    activity = np.zeros([ndim, ntp])
    dim1 = np.tile(period, int(ntp/ntp_period)+1)[:ntp]
    for i in range(ndim):
        activity[i] = np.roll(dim1, i*ntp_delay)

    if type(noise_std)!=0: #scitter spikes a bit
        noise_std_ntp = int(noise_std/dt)
        dim, time = np.nonzero(activity)

        noise_ind = np.random.normal(0, noise_std_ntp, time.size).astype(int)
        time += noise_ind #add noise to time points
    
        activity = np.zeros([ndim, ntp])
        activity[dim, time%ntp] = 1

    return activity
#ff_min = feedforward_generator(ndim=3, T = 100000, len_period=30, delay=10, dt=1, noise_std=3)
#ff_min_rate = inst_rate_alpha(ff_min, tau=tau, dt=dt) #inst_rate(ff_min, filter_width, dt)#


In [ ]:
# #save results:
# save = False

# if save:
#     results = {'trajectory':time_series, 'EPR_train':all_sig_train, 'EPR_test':all_sig_test} #TODO: add last weights
#     file_name = 'brunel50_av{}Hz_epr5runs_dt{}_alphaflt{}_norm__epochmax{}_dimh{}_numblocks{}'.format(mean_spike_rate, dt, tau, epoch_max, dim_h, num_blocks)
#     #if timeseries was mempot:
#     #file_name = 'big_brunel0_mempot_av{}Hz_epr5runs_dt{}_norm__epochmax{}_dimh{}_numblocks{}'.format(mean_spike_rate, dt, epoch_max, dim_h, num_blocks)
#     if Im_on_server:
#         path = cd + '/saved_results/network epr/Rajat epr/'
#     else:
#         path = cd + '\\saved_results\\network epr\\' 

#     np.save(path + file_name, results)